# 单臂重建对比（Version4 手搓逻辑 vs 官方 renderer 逻辑）

本 notebook 与原版功能相同，改正了原版所有变量命名中存在的语义混淆。

## 命名约定

全文统一使用 `T_target_from_source` 命名法，明确标注变换方向：

| 变量名 | 数学记号 | 含义 |
|--------|----------|------|
| `T_base_from_cam` | $T_{\text{base}\leftarrow\text{cam}}$ | camera POSE in real_base，由 `pose_in_link` 直接构造 |
| `T_cam_from_base` | $T_{\text{cam}\leftarrow\text{base}}$ | camera 外参 w.r.t. real_base，= inv(`T_base_from_cam`) |
| `cam_extrinsic_urdf` | $T_{\text{cam}\leftarrow\text{URDF\_base}}$ | renderer 所需外参，= `T_cam_from_base @ inv(ROBOT_PREDEFINED)` |

原版中 `cam_to_base_raw`、`cam_to_base`、`base_to_cam`、`cam_to_base_for_renderer` 分别对应上表各行，但命名在 POSE 与外参之间存在混淆；本版一律替换。

## 两套逻辑的关系

$$\text{renderer 逻辑} = O3D\_RENDER \times \text{手搓逻辑}$$

两套逻辑的场景几何完全相同，仅最终坐标系不同（手搓在相机系，renderer 在 O3D 渲染系，差一个 Y/Z 轴翻转）。

## 四元数约定

`pose_in_link` 中四元数顺序为 `[qw, qx, qy, qz]`（wxyz），直接传入 `transforms3d.quat2mat`，不做重排。


## 1) 导入与配置


In [ ]:
import os
import h5py
import numpy as np
import open3d as o3d
import k3d

from omegaconf import OmegaConf
from transforms3d.quaternions import quat2mat

from airexo.helpers.urdf_robot import forward_kinematic_single
from airexo.helpers.constants import ROBOT_PREDEFINED_TRANSFORMATION, O3D_RENDER_TRANSFORMATION

SCENE_PATH = "/data/haoxiang/data/FLIPPING_v3/train/scene_0001"
LOWDIM_H5_PATH = os.path.join(SCENE_PATH, "lowdim/lowdim.h5")
URDF_FILE = "airexo/urdf_models/robot/left_robot_inhand.urdf"
JOINT_CFG_PATH = "airexo/configs/joint/left/robot.yaml"

# pose_in_link 格式：[x, y, z, qw, qx, qy, qz]
CALIB_DATA = {
    "is_global": True,
    "pose_in_link": [
        0.07783932332093665,
        0.2078814260418823,
        0.34723683952957585,
        0.2273133855008057,
        -0.6785647482083789,
        0.6637673415778982,
        -0.21746591345696367,
    ],
    "error": 0.0024148397685646483,
    "parent_link_name": "world",
    "cam_serial": "104122060902",
    "intrinsics": [
        [915.384521484375, 0.0, 633.3715209960938],
        [0.0, 914.9421997070312, 354.1505432128906],
        [0.0, 0.0, 1.0],
    ],
}

print("Imports and config done.")
print(f"SCENE_PATH: {SCENE_PATH}")


## 2) 数据类 / 辅助函数

### `_process_calibration` 中三个变量的推导

`pose_in_link` 是 camera 在机器人 real_base 坐标系下的位姿，因此直接构造 $T_{\text{base}\leftarrow\text{cam}}$：

$$T_{\text{base}\leftarrow\text{cam}} = \begin{bmatrix} R_{\text{b}\leftarrow\text{c}} & t_{\text{cam in base}} \\ 0 & 1 \end{bmatrix}$$

其中平移列 $t_{\text{cam in base}}$ = `pose_in_link[:3]`（camera 原点在 base 系中的坐标），旋转矩阵由 `quat2mat(wxyz)` 得到。

三个派生量的关系链：

$$T_{\text{cam}\leftarrow\text{base}} = T_{\text{base}\leftarrow\text{cam}}^{-1}$$

$$\underbrace{T_{\text{cam}\leftarrow\text{URDF\_base}}}_{\texttt{cam\_extrinsic\_urdf}} = T_{\text{cam}\leftarrow\text{base}} \cdot ROBOT\_PREDEFINED^{-1}$$

`cam_extrinsic_urdf` 与 production 的 `calib_info.get_camera_to_base(real_base=False)` 数值完全相同。


In [ ]:
class SingleRobotData:
    """单臂机器人数据加载器"""

    def __init__(self, lowdim_h5_path, calib_data, joint_cfg_path, urdf_file):
        self.lowdim_h5_path = lowdim_h5_path
        self.calib_data = calib_data
        self.urdf_file = urdf_file
        self.joint_cfgs = OmegaConf.load(joint_cfg_path)
        self._load_lowdim_data()
        self._process_calibration()

    def _load_lowdim_data(self):
        with h5py.File(self.lowdim_h5_path, "r") as f:
            self.joint_positions = f["joint_position_rad_062046"][:]
            self.ee_states = f["ee_state_062046"][:]
            self.timestamps = f["timestamp"][:]
            if "tcp_pose_062046" in f:
                self.tcp_poses = f["tcp_pose_062046"][:]

        print(f"Lowdim loaded, frames: {len(self.timestamps)}")

    def _process_calibration(self):
        self.intrinsic = np.array(self.calib_data["intrinsics"], dtype=np.float32)
        pose_in_link = self.calib_data["pose_in_link"]

        position = np.array(pose_in_link[:3], dtype=np.float32)
        quaternion_wxyz = np.array(pose_in_link[3:], dtype=np.float32)
        rotation = quat2mat(quaternion_wxyz).astype(np.float32)

        # camera 在 real_base 系下的位姿，T_{base<-cam}
        self.T_base_from_cam = np.eye(4, dtype=np.float32)
        self.T_base_from_cam[:3, :3] = rotation
        self.T_base_from_cam[:3, 3] = position

        # camera 外参 w.r.t. real_base，T_{cam<-base}
        self.T_cam_from_base = np.linalg.inv(self.T_base_from_cam).astype(np.float32)

        # renderer 所需外参 w.r.t. URDF_base，等价于 get_camera_to_base(real_base=False)
        self.cam_extrinsic_urdf = (
            self.T_cam_from_base @ np.linalg.inv(ROBOT_PREDEFINED_TRANSFORMATION)
        ).astype(np.float32)

        print("Calibration done.")
        print("T_base_from_cam (camera POSE in real_base):")
        print(self.T_base_from_cam)
        print("T_cam_from_base (camera extrinsic w.r.t. real_base):")
        print(self.T_cam_from_base)
        print("cam_extrinsic_urdf (for renderer, w.r.t. URDF_base):")
        print(self.cam_extrinsic_urdf)

    def get_joint_at_timestamp(self, timestamp_idx):
        joint_angles = self.joint_positions[timestamp_idx]
        ee_state = self.ee_states[timestamp_idx, 0]
        return np.concatenate([joint_angles, [ee_state]], axis=0)


def reconstruct_fk_transforms_at_frame(robot_data, frame_idx):
    """正运动学，返回 transforms 与 visuals_map。"""
    joint_state = robot_data.get_joint_at_timestamp(frame_idx)
    transforms, visuals_map = forward_kinematic_single(
        joint=joint_state,
        joint_cfgs=robot_data.joint_cfgs,
        is_rad=True,
        urdf_file=robot_data.urdf_file,
        with_visuals_map=True,
    )
    return transforms, visuals_map


def pack_colors_to_uint32(colors_float):
    colors_u8 = (np.clip(colors_float, 0.0, 1.0) * 255).astype(np.uint32)
    return (colors_u8[:, 0] << 16) | (colors_u8[:, 1] << 8) | colors_u8[:, 2]


def manual_backproject_to_cam_points(rgb_path, depth_path, intrinsic, depth_scale=1000.0, depth_trunc=5.0, stride=2):
    """手动反投影，点云保持在相机坐标系。"""
    rgb = np.asarray(o3d.io.read_image(rgb_path))
    depth_raw = np.asarray(o3d.io.read_image(depth_path)).astype(np.float32)

    if rgb.ndim == 2:
        rgb = np.repeat(rgb[..., None], 3, axis=2)
    if rgb.shape[2] > 3:
        rgb = rgb[:, :, :3]

    depth = depth_raw / float(depth_scale)
    h, w = depth.shape

    fx, fy = float(intrinsic[0, 0]), float(intrinsic[1, 1])
    cx, cy = float(intrinsic[0, 2]), float(intrinsic[1, 2])

    uu, vv = np.meshgrid(np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32))

    valid = (depth > 0.0) & (depth < depth_trunc) & np.isfinite(depth)
    if stride > 1:
        valid &= (uu.astype(np.int32) % stride == 0) & (vv.astype(np.int32) % stride == 0)

    z = depth[valid]
    u = uu[valid]
    v = vv[valid]

    x = (u - cx) * z / fx
    y = (v - cy) * z / fy

    points_cam = np.stack([x, y, z], axis=1).astype(np.float32)
    colors = (rgb[valid].astype(np.float32) / 255.0)

    return points_cam, colors


## 3) Version4 手搓逻辑可视化（相机坐标系）

### 变换链

$$p_{\text{cam}} = \underbrace{T_{\text{cam}\leftarrow\text{base}}}_{\texttt{T\_cam\_from\_base}} \cdot \underbrace{ROBOT\_PREDEFINED}_{\text{URDF\_base}\to\text{real\_base}} \cdot \underbrace{T_{\text{FK}}}_{\text{link}\to\text{URDF\_base}} \cdot \underbrace{T_{\text{offset}}}_{\text{mesh}\to\text{link}} \cdot p_{\text{mesh}}$$

点云通过手动反投影已在相机坐标系中，mesh 经上述链也变换到相机坐标系，两者直接对齐。


In [ ]:
def visualize_version4_manual_k3d(robot_data, frame_idx, rgb_path, depth_path, subsample_step=2):
    """
    Version4 手搓逻辑：点云手动反投影保持在相机系，mesh 经 T_cam_from_base 变换到相机系。
    """
    points_cam, colors_cam = manual_backproject_to_cam_points(
        rgb_path=rgb_path,
        depth_path=depth_path,
        intrinsic=robot_data.intrinsic,
        depth_scale=1000.0,
        depth_trunc=5.0,
        stride=subsample_step,
    )

    transforms, visuals_map = reconstruct_fk_transforms_at_frame(robot_data, frame_idx)

    plot = k3d.plot(background_color=0xFFFFFF)
    plot += k3d.points(
        positions=points_cam,
        colors=pack_colors_to_uint32(colors_cam).astype(np.uint32),
        point_size=0.0025,
        shader="flat",
        name="Point Cloud (camera frame)",
    )

    urdf_dir = os.path.dirname(robot_data.urdf_file)
    mesh_count = 0

    for link, transform in transforms.items():
        visuals = visuals_map.get(link, [])
        for visual in visuals:
            if visual.geom_param is None:
                continue

            mesh_rel = visual.geom_param[0] if isinstance(visual.geom_param, (list, tuple)) else visual.geom_param
            mesh_path = os.path.join(urdf_dir, str(mesh_rel))
            if not os.path.exists(mesh_path):
                continue

            mesh_o3d = o3d.io.read_triangle_mesh(mesh_path)

            # mesh局部系 -> link系 -> URDF_base系 -> real_base系 -> cam系
            tf = (
                robot_data.T_cam_from_base
                @ ROBOT_PREDEFINED_TRANSFORMATION
                @ transform.matrix()
                @ visual.offset.matrix()
            )
            mesh_o3d.transform(tf)

            verts = np.asarray(mesh_o3d.vertices).astype(np.float32)
            faces = np.asarray(mesh_o3d.triangles).astype(np.uint32)
            if len(verts) == 0 or len(faces) == 0:
                continue

            plot += k3d.mesh(
                vertices=verts,
                indices=faces,
                color=0x4A90E2,
                opacity=0.65,
                name=f"V4:{link}:{os.path.basename(str(mesh_rel))}",
            )
            mesh_count += 1

    axis_size = 0.2
    plot += k3d.vectors(
        origins=[[0, 0, 0], [0, 0, 0], [0, 0, 0]],
        vectors=[[axis_size, 0, 0], [0, axis_size, 0], [0, 0, axis_size]],
        colors=[0xFF0000, 0x00FF00, 0x0000FF],
        line_width=0.01,
    )

    plot.display()
    print(f"Version4 done: {len(points_cam)} pcd points, {mesh_count} meshes (camera frame)")
    return plot


## 4) 官方 renderer 逻辑可视化（O3D 渲染空间）

### 变换链

$$p_{\text{O3D}} = O3D\_RENDER \cdot \underbrace{T_{\text{cam}\leftarrow\text{URDF\_base}}}_{\texttt{cam\_extrinsic\_urdf}} \cdot ROBOT\_PREDEFINED \cdot T_{\text{FK}} \cdot T_{\text{offset}} \cdot p_{\text{mesh}}$$

代数展开后等价于：

$$= O3D\_RENDER \cdot T_{\text{cam}\leftarrow\text{base}} \cdot T_{\text{FK}} \cdot T_{\text{offset}} \cdot p_{\text{mesh}}$$

因为 `cam_extrinsic_urdf = T_cam_from_base @ inv(ROBOT_PREDEFINED)`，与右侧的 `ROBOT_PREDEFINED` 相乘后 inv 抵消。点云经 `pcd.transform(O3D_RENDER)` 直接从相机系变换到 O3D 渲染系，与 mesh 对齐。


In [ ]:
def visualize_renderer_official_k3d(robot_data, frame_idx, rgb_path, depth_path, cam_extrinsic_urdf, subsample_step=4):
    """
    官方 renderer 逻辑：点云经 O3D_RENDER 变换到渲染系，mesh 经完整变换链到相同渲染系。
    """
    rgb_img = o3d.io.read_image(rgb_path)
    depth_img = o3d.io.read_image(depth_path)

    rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
        rgb_img,
        depth_img,
        depth_scale=1000.0,
        convert_rgb_to_intensity=False,
    )

    K = robot_data.intrinsic
    h, w = np.asarray(rgb_img).shape[:2]
    intrinsic_o3d = o3d.camera.PinholeCameraIntrinsic(
        width=int(w),
        height=int(h),
        fx=float(K[0, 0]),
        fy=float(K[1, 1]),
        cx=float(K[0, 2]),
        cy=float(K[1, 2]),
    )

    # 点云从相机系变换到 O3D 渲染系（翻转 Y/Z 轴）
    pcd = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, intrinsic_o3d)
    pcd.transform(O3D_RENDER_TRANSFORMATION)

    pcd_points = np.asarray(pcd.points).astype(np.float32)
    pcd_colors = np.asarray(pcd.colors).astype(np.float32)

    valid = np.isfinite(pcd_points).all(axis=1)
    pcd_points = pcd_points[valid]
    pcd_colors = pcd_colors[valid]

    if subsample_step > 1:
        pcd_points = pcd_points[::subsample_step]
        pcd_colors = pcd_colors[::subsample_step]

    transforms, visuals_map = reconstruct_fk_transforms_at_frame(robot_data, frame_idx)

    plot = k3d.plot(background_color=0xFFFFFF)
    plot += k3d.points(
        positions=pcd_points,
        colors=pack_colors_to_uint32(pcd_colors).astype(np.uint32),
        point_size=0.0025,
        shader="flat",
        name="Point Cloud (O3D render space)",
    )

    urdf_dir = os.path.dirname(robot_data.urdf_file)
    mesh_count = 0

    for link, transform in transforms.items():
        visuals = visuals_map.get(link, [])
        for visual in visuals:
            if visual.geom_param is None:
                continue

            mesh_rel = visual.geom_param[0] if isinstance(visual.geom_param, (list, tuple)) else visual.geom_param
            mesh_path = os.path.join(urdf_dir, str(mesh_rel))
            if not os.path.exists(mesh_path):
                continue

            mesh_o3d = o3d.io.read_triangle_mesh(mesh_path)

            # mesh局部系 -> URDF_base系 -> cam系 -> O3D渲染系，RPRED与cam_extrinsic_urdf中的inv(RPRED)代数抵消
            tf = (
                O3D_RENDER_TRANSFORMATION
                @ cam_extrinsic_urdf
                @ ROBOT_PREDEFINED_TRANSFORMATION
                @ transform.matrix()
                @ visual.offset.matrix()
            )
            mesh_o3d.transform(tf)

            verts = np.asarray(mesh_o3d.vertices).astype(np.float32)
            faces = np.asarray(mesh_o3d.triangles).astype(np.uint32)
            if len(verts) == 0 or len(faces) == 0:
                continue

            plot += k3d.mesh(
                vertices=verts,
                indices=faces,
                color=0xFF4444,
                opacity=0.65,
                name=f"Renderer:{link}:{os.path.basename(str(mesh_rel))}",
            )
            mesh_count += 1

    axis_size = 0.2
    plot += k3d.vectors(
        origins=[[0, 0, 0], [0, 0, 0], [0, 0, 0]],
        vectors=[[axis_size, 0, 0], [0, axis_size, 0], [0, 0, axis_size]],
        colors=[0xFF0000, 0x00FF00, 0x0000FF],
        line_width=0.01,
    )

    plot.display()
    print(f"Renderer done: {len(pcd_points)} pcd points, {mesh_count} meshes (O3D render space)")
    return plot


## 5) 运行示例


In [ ]:
robot_data = SingleRobotData(
    lowdim_h5_path=LOWDIM_H5_PATH,
    calib_data=CALIB_DATA,
    joint_cfg_path=JOINT_CFG_PATH,
    urdf_file=URDF_FILE,
)

FRAME_IDX = 0
rgb_path   = "/data/haoxiang/data/FLIPPING_v3/train/scene_0001/cam_104122060902/color/1767593840262.png"
depth_path = "/data/haoxiang/data/FLIPPING_v3/train/scene_0001/cam_104122060902/depth/1767593840262.png"

plot_version4 = visualize_version4_manual_k3d(
    robot_data=robot_data,
    frame_idx=FRAME_IDX,
    rgb_path=rgb_path,
    depth_path=depth_path,
    subsample_step=2,
)

plot_renderer = visualize_renderer_official_k3d(
    robot_data=robot_data,
    frame_idx=FRAME_IDX,
    rgb_path=rgb_path,
    depth_path=depth_path,
    cam_extrinsic_urdf=robot_data.cam_extrinsic_urdf,
    subsample_step=4,
)
